# Business Analytics – Validation and Analysis

Use this notebook after the CTAS table has been created and registered.

**Table:** `retail_gold_db.gld_business_analytics`

In [ ]:
-- 1. List Gold tables to confirm the analytics table is registered
SHOW TABLES IN retail_gold_db;

In [ ]:
-- 2. Count rows
SELECT COUNT(*) AS total_rows
FROM retail_gold_db.gld_business_analytics;

In [ ]:
-- 3. Inspect output rows
SELECT *
FROM retail_gold_db.gld_business_analytics
LIMIT 50;

In [ ]:
-- 4. Overall business metrics
SELECT
    COUNT(*) AS total_order_items,
    COUNT(DISTINCT transaction_id) AS total_orders,
    SUM(quantity) AS total_units_sold,
    SUM(net_amount) AS total_revenue,
    SUM(net_amount_inr) AS total_revenue_inr,
    AVG(net_amount_inr) AS average_item_value
FROM retail_gold_db.gld_business_analytics;

In [ ]:
-- 5. Monthly revenue
SELECT
    year,
    month_name,
    SUM(net_amount_inr) AS monthly_revenue
FROM retail_gold_db.gld_business_analytics
GROUP BY year, month_name
ORDER BY year, MIN(transaction_date);

In [ ]:
-- 6. Revenue by category
SELECT
    category_name,
    SUM(net_amount_inr) AS total_revenue,
    SUM(quantity) AS units_sold
FROM retail_gold_db.gld_business_analytics
GROUP BY category_name
ORDER BY total_revenue DESC;

In [ ]:
-- 7. Revenue by brand
SELECT
    brand_name,
    SUM(net_amount_inr) AS total_revenue,
    COUNT(DISTINCT transaction_id) AS total_orders
FROM retail_gold_db.gld_business_analytics
GROUP BY brand_name
ORDER BY total_revenue DESC;

In [ ]:
-- 8. Revenue by sales channel
SELECT
    channel,
    SUM(net_amount_inr) AS total_revenue,
    COUNT(DISTINCT transaction_id) AS total_orders,
    SUM(quantity) AS units_sold
FROM retail_gold_db.gld_business_analytics
GROUP BY channel
ORDER BY total_revenue DESC;

In [ ]:
-- 9. Hourly sales analysis
SELECT
    hour_of_day,
    COUNT(DISTINCT transaction_id) AS total_orders,
    SUM(net_amount_inr) AS total_revenue
FROM retail_gold_db.gld_business_analytics
GROUP BY hour_of_day
ORDER BY hour_of_day;

In [ ]:
-- 10. Weekday versus weekend
SELECT
    CASE
        WHEN is_weekend = 1 THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_type,
    COUNT(DISTINCT transaction_id) AS total_orders,
    SUM(quantity) AS units_sold,
    SUM(net_amount_inr) AS total_revenue
FROM retail_gold_db.gld_business_analytics
GROUP BY is_weekend;

## Metric interpretation

`AVG(net_amount_inr)` is the average value per order-item row, not average order value. For average order value, first aggregate revenue by `transaction_id`, then average those order totals.

In [ ]:
-- 11. Average order value (AOV), calculated at order level
WITH order_totals AS (
    SELECT
        transaction_id,
        SUM(net_amount_inr) AS order_revenue_inr
    FROM retail_gold_db.gld_business_analytics
    GROUP BY transaction_id
)
SELECT AVG(order_revenue_inr) AS average_order_value_inr
FROM order_totals;